# Sentiment-Enhanced Prediction Market Modeling

**Question:** Do aggregated Twitter sentiment signals about a political event improve prediction of short-term movements in prediction market probabilities?

## Expected Input Files

Run the scripts in `scripts/` to get the following files in `data/raw/` before running the modeling sections:

`market_data.csv`
`tweets_data.csv`



In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

In [ ]:
ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
FIG_DIR = ROOT / "outputs" / "figures"

for path in [RAW_DIR, PROCESSED_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

MARKET_PATH = RAW_DIR / "market_data-KH.csv"
TWEETS_PATH = RAW_DIR / "tweets_data.csv"

# The tweet export has daily timestamps, so model at daily frequency.
AGG_FREQ = "1D"
HORIZONS = [1, 3, 7]
EVENT_ID = "election_2024"
MARKET_CUTOFF = pd.Timestamp("2024-11-06", tz="UTC")

(PosixPath('/Users/izhu/sentiment-market-prediction/data/raw'),
 PosixPath('/Users/izhu/sentiment-market-prediction/data/processed'),
 PosixPath('/Users/izhu/sentiment-market-prediction/outputs/figures'))

In [ ]:
SENTIMENT_MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
SENTIMENT_BATCH_SIZE = 32
SENTIMENT_CHUNK_SIZE = 1024

_sentiment_pipeline = None

def get_sentiment_pipeline():
    import os

    global _sentiment_pipeline
    if _sentiment_pipeline is None:
        os.environ.setdefault("DISABLE_SAFETENSORS_CONVERSION", "1")
        try:
            from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
        except ImportError as exc:
            raise ImportError(
                "DistilBERT sentiment scoring requires transformers and torch. "
                "Install project dependencies with `pip install -r requirements.txt`."
            ) from exc

        tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(
            SENTIMENT_MODEL_NAME,
            use_safetensors=False,
        )
        _sentiment_pipeline = pipeline(
            "sentiment-analysis",
            model=model,
            tokenizer=tokenizer,
            top_k=None,
        )
    return _sentiment_pipeline

def _score_from_bert_labels(label_scores: list[dict]):
    normalized = {item["label"].upper(): float(item["score"]) for item in label_scores}
    positive = normalized.get("POS", normalized.get("POSITIVE", 0.0))
    negative = normalized.get("NEG", normalized.get("NEGATIVE", 0.0))
    top = max(label_scores, key=lambda item: item["score"])
    return positive - negative, top["label"], float(top["score"])

def score_texts_with_bert(texts: pd.Series):
    clean_texts = texts.fillna("").astype(str)
    results = pd.DataFrame(
        {
            "sentiment_score": np.zeros(len(clean_texts), dtype=float),
            "sentiment_label": ["NEU"] * len(clean_texts),
            "sentiment_confidence": np.zeros(len(clean_texts), dtype=float),
        },
        index=clean_texts.index,
    )
    non_empty = clean_texts.str.strip().ne("")
    if not non_empty.any():
        return results

    sentiment_pipe = get_sentiment_pipeline()
    texts_to_score = clean_texts.loc[non_empty]
    scored_chunks = []

    for start in range(0, len(texts_to_score), SENTIMENT_CHUNK_SIZE):
        chunk = texts_to_score.iloc[start : start + SENTIMENT_CHUNK_SIZE]
        model_outputs = sentiment_pipe(
            chunk.tolist(),
            batch_size=SENTIMENT_BATCH_SIZE,
            truncation=True,
        )

        rows = []
        for output in model_outputs:
            label_scores = output if isinstance(output, list) else [output]
            score, label, confidence = _score_from_bert_labels(label_scores)
            rows.append(
                {
                    "sentiment_score": score,
                    "sentiment_label": label,
                    "sentiment_confidence": confidence,
                }
            )
        scored_chunks.append(pd.DataFrame(rows, index=chunk.index))

    scored = pd.concat(scored_chunks).sort_index()
    results.loc[scored.index, ["sentiment_score", "sentiment_label", "sentiment_confidence"]] = scored
    return results

def load_market_data(path: Path, event_id=None):
    df = pd.read_csv(path)
    required = {"timestamp", "market_prob"}

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp", "market_prob"]).sort_values("timestamp").copy()

    if event_id is not None and "event_id" in df.columns:
        df = df.loc[df["event_id"] == event_id].copy()

    df["market_prob"] = pd.to_numeric(df["market_prob"], errors="coerce")
    df = df.dropna(subset=["market_prob"])
    return df

def load_tweets_data(path: Path, event_id=None):
    df = pd.read_csv(path)
    required = {"timestamp", "text"}
    
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp"]).sort_values("timestamp").copy()

    if event_id is not None and "event_id" in df.columns:
        df = df.loc[df["event_id"] == event_id].copy()

    return df

def add_sentiment_scores(tweets: pd.DataFrame):
    tweets = tweets.copy()
    sentiment = score_texts_with_bert(tweets["text"])
    tweets = tweets.join(sentiment)
    tweets["sentiment_score"] = pd.to_numeric(tweets["sentiment_score"], errors="coerce").fillna(0.0)
    tweets["tweet_count"] = 1
    return tweets

def aggregate_tweets(tweets: pd.DataFrame, freq: str = "1D"):
    tweets = tweets.copy()
    tweets["sentiment_score"] = pd.to_numeric(tweets["sentiment_score"], errors="coerce").fillna(0.0)
    tweets["tweet_count"] = pd.to_numeric(tweets["tweet_count"], errors="coerce").fillna(0).astype(int)
    tweets["bucket"] = tweets["timestamp"].dt.floor(freq)

    if tweets.empty:
        return pd.DataFrame(
            columns=[
                "timestamp",
                "sentiment_mean",
                "sentiment_std",
                "sentiment_sum",
                "tweet_count",
                "sentiment_momentum",
                "tweet_count_change",
            ]
        )

    agg = (
        tweets.groupby("bucket")
        .agg(
            sentiment_mean=("sentiment_score", "mean"),
            sentiment_std=("sentiment_score", "std"),
            sentiment_sum=("sentiment_score", "sum"),
            tweet_count=("tweet_count", "sum"),
        )
        .reset_index()
        .rename(columns={"bucket": "timestamp"})
    )

    agg["sentiment_std"] = agg["sentiment_std"].fillna(0.0)
    agg["sentiment_momentum"] = agg["sentiment_mean"].diff()
    agg["tweet_count_change"] = agg["tweet_count"].diff()
    return agg

def build_modeling_frame(market: pd.DataFrame, tweet_features: pd.DataFrame, freq: str = "1D"):
    market = market.copy()
    market["timestamp"] = market["timestamp"].dt.floor(freq)

    market_agg = (
        market.groupby("timestamp")
        .agg(
            market_prob=("market_prob", "last"),
            market_volume=("market_volume", "last") if "market_volume" in market.columns else ("market_prob", "size"),
        )
        .reset_index()
    )

    frame = market_agg.merge(tweet_features, on="timestamp", how="left").sort_values("timestamp").copy()

    fill_zero_cols = [
        "sentiment_mean", "sentiment_std", "sentiment_sum",
        "tweet_count", "sentiment_momentum", "tweet_count_change"
    ]
    for col in fill_zero_cols:
        if col in frame.columns:
            frame[col] = frame[col].fillna(0.0)

    frame["market_return_1"] = frame["market_prob"].diff(1)
    frame["market_return_3"] = frame["market_prob"].diff(3)
    frame["market_return_6"] = frame["market_prob"].diff(6)
    frame["market_volatility_6"] = frame["market_prob"].rolling(6).std()
    frame["market_ma_3"] = frame["market_prob"].rolling(3).mean()
    frame["market_ma_6"] = frame["market_prob"].rolling(6).mean()
    frame["tweet_count_log1p"] = np.log1p(frame["tweet_count"])
    return frame

def add_targets(frame: pd.DataFrame, horizon: int):
    df = frame.copy()
    future_prob = df["market_prob"].shift(-horizon)
    target_change = future_prob - df["market_prob"]
    df[f"target_change_h{horizon}"] = target_change
    df[f"target_direction_h{horizon}"] = np.where(target_change.notna(), target_change > 0, np.nan)
    return df

def prepare_xy(df: pd.DataFrame, features: list[str], target: str):
    clean = df.dropna(subset=features + [target]).copy()
    X = clean[features]
    y = clean[target]
    return X, y

In [ ]:
market_df = load_market_data(MARKET_PATH, event_id=EVENT_ID)
tweets_df = load_tweets_data(TWEETS_PATH, event_id=EVENT_ID)

if MARKET_CUTOFF is not None:
    market_df = market_df.loc[market_df["timestamp"] < MARKET_CUTOFF].copy()
    tweets_df = tweets_df.loc[tweets_df["timestamp"] < MARKET_CUTOFF].copy()

tweets_scored = add_sentiment_scores(tweets_df)
tweet_features = aggregate_tweets(tweets_scored, freq=AGG_FREQ)
model_df = build_modeling_frame(market_df, tweet_features, freq=AGG_FREQ)

print("Market rows:", len(market_df))
print("Tweet rows:", len(tweets_df))
print("Modeling rows:", len(model_df))
model_df.head()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
plot_df = model_df.copy()
plot_df["timestamp_plot"] = plot_df["timestamp"].dt.tz_convert(None)
recent_df = plot_df.tail(80)

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

sns.lineplot(data=plot_df, x="timestamp_plot", y="market_prob", ax=axes[0], color="#1f77b4")
axes[0].set_title("Prediction Market Probability")

sns.lineplot(data=plot_df, x="timestamp_plot", y="sentiment_mean", ax=axes[1], color="#d62728")
axes[1].set_title("Aggregated Sentiment Mean")

axes[2].bar(recent_df["timestamp_plot"], recent_df["tweet_count"], width=0.8, color="#2ca02c")
axes[2].set_title("Recent Tweet Volume")
axes[2].set_xlabel("timestamp")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
baseline_features = [
    "market_prob",
    "market_return_1",
    "market_return_3",
    "market_return_6",
    "market_volatility_6",
    "market_ma_3",
    "market_ma_6",
    "market_volume",
]

sentiment_features = baseline_features + [
    "sentiment_mean",
    "sentiment_std",
    "sentiment_sum",
    "tweet_count",
    "tweet_count_log1p",
    "sentiment_momentum",
    "tweet_count_change",
]

preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

regression_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", LinearRegression()),
    ]
)

classification_pipeline = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

def evaluate_time_series_model(X, y, estimator, task="regression", splits=5):
    n_splits = min(splits, max(2, len(X) // 5))
    splitter = TimeSeriesSplit(n_splits=n_splits)
    fold_rows = []

    for fold, (train_idx, test_idx) in enumerate(splitter.split(X), start=1):
        model = clone(estimator)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if task == "classification" and y_train.nunique() < 2:
            pred_labels = np.repeat(y_train.iloc[0], len(y_test))
            fold_rows.append(
                {
                    "fold": fold,
                    "accuracy": accuracy_score(y_test, pred_labels),
                    "f1": f1_score(y_test, pred_labels, zero_division=0),
                }
            )
            continue

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        if task == "regression":
            fold_rows.append(
                {
                    "fold": fold,
                    "mse": mean_squared_error(y_test, preds),
                    "mae": mean_absolute_error(y_test, preds),
                    "r2": r2_score(y_test, preds),
                }
            )
        else:
            pred_labels = (preds >= 0.5).astype(int) if preds.ndim == 1 else np.argmax(preds, axis=1)
            fold_rows.append(
                {
                    "fold": fold,
                    "accuracy": accuracy_score(y_test, pred_labels),
                    "f1": f1_score(y_test, pred_labels, zero_division=0),
                }
            )

    return pd.DataFrame(fold_rows)

In [ ]:
experiment_rows = []

for horizon in HORIZONS:
    horizon_df = add_targets(model_df, horizon=horizon)

    X_base_reg, y_reg = prepare_xy(horizon_df, baseline_features, f"target_change_h{horizon}")
    X_aug_reg, _ = prepare_xy(horizon_df, sentiment_features, f"target_change_h{horizon}")

    reg_base_scores = evaluate_time_series_model(X_base_reg, y_reg, regression_pipeline, task="regression")
    reg_aug_scores = evaluate_time_series_model(X_aug_reg, y_reg.loc[X_aug_reg.index], regression_pipeline, task="regression")

    for label, scores in [("baseline", reg_base_scores), ("sentiment_augmented", reg_aug_scores)]:
        experiment_rows.append(
            {
                "horizon_days": horizon,
                "task": "regression",
                "model_family": label,
                "mse_mean": scores["mse"].mean(),
                "mae_mean": scores["mae"].mean(),
                "r2_mean": scores["r2"].mean(),
            }
        )

    X_base_clf, y_clf = prepare_xy(horizon_df, baseline_features, f"target_direction_h{horizon}")
    X_aug_clf, _ = prepare_xy(horizon_df, sentiment_features, f"target_direction_h{horizon}")

    clf_base_scores = evaluate_time_series_model(X_base_clf, y_clf, classification_pipeline, task="classification")
    clf_aug_scores = evaluate_time_series_model(X_aug_clf, y_clf.loc[X_aug_clf.index], classification_pipeline, task="classification")

    for label, scores in [("baseline", clf_base_scores), ("sentiment_augmented", clf_aug_scores)]:
        experiment_rows.append(
            {
                "horizon_days": horizon,
                "task": "classification",
                "model_family": label,
                "accuracy_mean": scores["accuracy"].mean(),
                "f1_mean": scores["f1"].mean(),
            }
        )

results_df = pd.DataFrame(experiment_rows)
results_df

In [ ]:
regression_results = results_df.query("task == 'regression'").copy()
classification_results = results_df.query("task == 'classification'").copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=regression_results,
    x="horizon_days",
    y="mse_mean",
    hue="model_family",
    ax=axes[0],
    palette=["#4c78a8", "#f58518"],
)
axes[0].set_title("Regression Comparison by Horizon")
axes[0].set_ylabel("Mean CV MSE")

sns.barplot(
    data=classification_results,
    x="horizon_days",
    y="f1_mean",
    hue="model_family",
    ax=axes[1],
    palette=["#4c78a8", "#f58518"],
)
axes[1].set_title("Classification Comparison by Horizon")
axes[1].set_ylabel("Mean CV F1")

plt.tight_layout()
plt.show()

In [ ]:
def lead_lag_analysis(frame: pd.DataFrame, max_lag: int = 7):
    rows = []
    base = frame[["timestamp", "sentiment_mean", "market_return_1"]].dropna().copy()

    for lag in range(-max_lag, max_lag + 1):
        shifted = base["sentiment_mean"].shift(lag)
        corr = shifted.corr(base["market_return_1"])
        rows.append({"lag_days": lag, "correlation": corr})

    return pd.DataFrame(rows)

lag_df = lead_lag_analysis(model_df, max_lag=7)

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=lag_df, x="lag_days", y="correlation", marker="o", color="#6f4e7c")
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title("Lead/Lag Correlation: Sentiment vs 1-Step Market Return")
plt.xlabel("Lag in Days (negative means sentiment leads)")
plt.ylabel("Correlation")
plt.tight_layout()
plt.show()

In [ ]:
results_path = PROCESSED_DIR / "model_comparison_results.csv"
lag_path = PROCESSED_DIR / "lead_lag_results.csv"

results_df.to_csv(results_path, index=False)
lag_df.to_csv(lag_path, index=False)

print("Saved:", results_path)
print("Saved:", lag_path)